## TC 5033
## Deep Learning
## Transformers

#### Activity 4: Implementing a Translator

- Objective

To understand the Transformer Architecture by Implementing a translator.

- Instructions

    This activity requires submission in teams. While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Follow the provided code. The code already implements a transformer from scratch as explained in one of [week's 9 videos](https://youtu.be/XefFj4rLHgU)

    Since the provided code already implements a simple translator, your job for this assignment is to understand it fully, and document it using pictures, figures, and markdown cells.  You should test your translator with at least 10 sentences. The dataset used for this task was obtained from [Tatoeba, a large dataset of sentences and translations](https://tatoeba.org/en/downloads).
  
- Evaluation Criteria

    - Code Readability and Comments
    - Traning a translator
    - Translating at least 10 sentences.

- Submission

Submit this Jupyter Notebook in canvas with your complete solution, ensuring your code is well-commented and includes Markdown cells that explain your design choices, results, and any challenges you encountered.



# Team 48

* Leonardo Daniel Rodriguez Vega - A01797465
* Rigoberto Bracamontes Salazar - A01134473
* Mayra Judith Vargas Rivero - A01797375
* Oliver Jordy Perez Escamilla - A01797471

#### Script to convert csv to text file

## Transformer - Attention is all you need

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np
import re

torch.manual_seed(23)

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')  # Use Apple Silicon GPU!
else:
    device = torch.device('cpu')

print(device)

mps


In [ ]:
MAX_SEQ_LEN = 128

## Transformer Architecture Overview

The Transformer, introduced in ["Attention Is All You Need" (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762), revolutionized sequence-to-sequence modeling by replacing recurrence entirely with **self-attention** mechanisms. Unlike RNNs/LSTMs, which process tokens sequentially, the Transformer processes all tokens in parallel, leading to faster training and better capture of long-range dependencies.

<img src="https://upload.wikimedia.org/wikipedia/commons/8/8f/The-Transformer-model-architecture.png" width="350" alt="Transformer Architecture"/>

*Figure: The Transformer model architecture (Vaswani et al., 2017). The encoder (left) processes the source sentence; the decoder (right) generates the target sentence autoregressively.*

### Key Components

| Component | Purpose |
|-----------|---------|
| **Positional Embedding** | Injects position information since self-attention is position-agnostic |
| **Multi-Head Attention** | Allows the model to attend to different representation subspaces simultaneously |
| **Feed-Forward Network** | Applies non-linear transformations independently to each position |
| **Layer Normalization** | Stabilizes training by normalizing activations |
| **Residual Connections** | Facilitates gradient flow through deep networks |
| **Masking** | Padding mask (ignores `<pad>` tokens) and causal mask (prevents decoder from looking ahead) |

### Positional Encoding

Since the Transformer has no recurrence, sinusoidal functions at different frequencies inject position information:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right) \quad\quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

### Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Scaling by $\sqrt{d_k}$ prevents dot products from becoming too large, which would push softmax into regions with very small gradients.

### Multi-Head Attention

Instead of computing a single attention, the model uses $h=8$ parallel heads, each operating on a $d_k = d_{model}/h = 64$ dimensional subspace:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \cdot W^O$$

### Hyperparameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| `d_model` | 512 | Embedding dimension |
| `num_heads` | 8 | Number of attention heads |
| `d_ff` | 2048 | Feed-forward hidden dimension |
| `num_layers` | 6 | Number of encoder/decoder layers |
| `dropout` | 0.1 | Dropout rate |
| `MAX_SEQ_LEN` | 128 | Maximum sequence length |

In [ ]:
class PositionalEmbedding(nn.Module):

    def __init__(self, d_model, max_seq_len = MAX_SEQ_LEN):
        super().__init__()
        self.pos_embed_matrix = torch.zeros(max_seq_len, d_model, device=device)
        # Position indices [0, 1, ..., max_seq_len-1] as column vector
        token_pos = torch.arange(0, max_seq_len, dtype = torch.float).unsqueeze(1)
        # Denominator term: 10000^(2i/d_model), computed in log-space for numerical stability
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0)/d_model))
        # Even indices get sin, odd indices get cos
        self.pos_embed_matrix[:, 0::2] = torch.sin(token_pos * div_term)
        self.pos_embed_matrix[:, 1::2] = torch.cos(token_pos * div_term)
        # Reshape to [max_seq_len, 1, d_model] for broadcasting with batched inputs
        self.pos_embed_matrix = self.pos_embed_matrix.unsqueeze(0).transpose(0,1)

    def forward(self, x):
        # Add positional encoding to input embeddings, truncated to actual sequence length
        return x + self.pos_embed_matrix[:x.size(0), :]

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model = 512, num_heads = 8):
        super().__init__()
        assert d_model % num_heads == 0, 'Embedding size not compatible with num heads'

        # d_k = d_v = d_model / num_heads (each head operates on a subspace)
        self.d_v = d_model // num_heads
        self.d_k = self.d_v
        self.num_heads = num_heads

        # Learned linear projections for Q, K, V and the output concatenation
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask = None):
        batch_size = Q.size(0)
        # Project and reshape into multiple heads:
        # [batch, seq_len, d_model] -> [batch, num_heads, seq_len, d_k]
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )

        weighted_values, attention = self.scale_dot_product(Q, K, V, mask)
        # Concatenate heads: [batch, num_heads, seq_len, d_k] -> [batch, seq_len, d_model]
        weighted_values = weighted_values.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads*self.d_k)
        # Final linear projection
        weighted_values = self.W_o(weighted_values)

        return weighted_values, attention


    def scale_dot_product(self, Q, K, V, mask = None):
        # Attention(Q,K,V) = softmax(Q*K^T / sqrt(d_k)) * V
        # Scaling by sqrt(d_k) prevents dot products from growing too large
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # Fill masked positions with -inf so softmax assigns ~0 probability
            scores = scores.masked_fill(mask == 0, -1e9)
        attention = F.softmax(scores, dim = -1)
        weighted_values = torch.matmul(attention, V)

        return weighted_values, attention


class PositionFeedForward(nn.Module):

    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderSubLayer(nn.Module):

    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.droupout1 = nn.Dropout(dropout)
        self.droupout2 = nn.Dropout(dropout)

    def forward(self, x, mask = None):
        # Self-attention: Q=K=V=x (the encoder attends to itself)
        attention_score, _ = self.self_attn(x, x, x, mask)
        x = x + self.droupout1(attention_score)  # Residual connection
        x = self.norm1(x)
        x = x + self.droupout2(self.ffn(x))      # Residual connection
        return self.norm2(x)

class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderSubLayer(nn.Module):

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, target_mask=None, encoder_mask=None):
        # 1. Masked self-attention (causal mask prevents looking at future tokens)
        attention_score, _ = self.self_attn(x, x, x, target_mask)
        x = x + self.dropout1(attention_score)
        x = self.norm1(x)

        # 2. Cross-attention: Q from decoder, K and V from encoder output
        encoder_attn, _ = self.cross_attn(x, encoder_output, encoder_output, encoder_mask)
        x = x + self.dropout2(encoder_attn)
        x = self.norm2(x)

        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        return self.norm3(x)

class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, target_mask, encoder_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, target_mask, encoder_mask)
        return self.norm(x)

### The Complete Transformer

The `Transformer` class ties together the encoder and decoder with their respective embedding layers and a final linear projection:

**Forward pass pipeline:**
1. **Masking**: Generate a padding mask for the source and a combined padding + causal mask for the target
2. **Source embedding**: Token embedding scaled by $\sqrt{d_{model}}$ + positional encoding
3. **Encoder**: Process the source through N=6 encoder layers
4. **Target embedding**: Same scaling and positional encoding for the target
5. **Decoder**: Process the target through N=6 decoder layers, using cross-attention to the encoder output
6. **Output projection**: Linear layer maps decoder output to target vocabulary logits

**Masking explained:**
- **Source (padding) mask**: A boolean tensor that is `True` for real tokens and `False` for `<pad>` positions, preventing attention to padding
- **Target (causal) mask**: Combines the padding mask with a lower-triangular matrix, ensuring that position $i$ can only attend to positions $\leq i$ (autoregressive property)

In [ ]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers,
                 input_vocab_size, target_vocab_size,
                 max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.encoder_embedding = nn.Embedding(input_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.pos_embedding = PositionalEmbedding(d_model, max_len)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.output_layer = nn.Linear(d_model, target_vocab_size)

    def forward(self, source, target):
        # Generate padding mask for source and causal mask for target
        source_mask, target_mask = self.mask(source, target)

        # Encoder pipeline: embed -> scale by sqrt(d_model) -> add positional encoding -> encode
        source = self.encoder_embedding(source) * math.sqrt(self.encoder_embedding.embedding_dim)
        source = self.pos_embedding(source)
        encoder_output = self.encoder(source, source_mask)

        # Decoder pipeline: embed -> scale -> positional encoding -> decode with cross-attention
        target = self.decoder_embedding(target) * math.sqrt(self.decoder_embedding.embedding_dim)
        target = self.pos_embedding(target)
        output = self.decoder(target, encoder_output, target_mask, source_mask)

        # Project decoder output to target vocabulary logits
        return self.output_layer(output)



    def mask(self, source, target):
        # Padding mask: True where token is NOT padding (id != 0)
        source_mask = (source != 0).unsqueeze(1).unsqueeze(2)
        target_mask = (target != 0).unsqueeze(1).unsqueeze(2)
        # Causal (look-ahead) mask: lower triangular matrix prevents
        # attending to future positions during autoregressive decoding
        size = target.size(1)
        no_mask = torch.tril(torch.ones((1, size, size), device=device)).bool()
        # Combine padding mask with causal mask
        target_mask = target_mask & no_mask
        return source_mask, target_mask


#### Simple test

In [ ]:
seq_len_source = 10
seq_len_target = 10
batch_size = 2
input_vocab_size = 50
target_vocab_size = 50

source = torch.randint(1, input_vocab_size, (batch_size, seq_len_source))
target = torch.randint(1, target_vocab_size, (batch_size, seq_len_target))

In [ ]:
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6

model = Transformer(d_model, num_heads, d_ff, num_layers,
                  input_vocab_size, target_vocab_size,
                  max_len=MAX_SEQ_LEN, dropout=0.1)

model = model.to(device)
source = source.to(device)
target = target.to(device)

In [ ]:
output = model(source, target)

In [ ]:
# Expected output shape -> [batch, seq_len_target, target_vocab_size] i.e. [2, 10, 50]
print(f'ouput.shape {output.shape}')

ouput.shape torch.Size([2, 10, 50])


### Translator: English to Spanish

#### Data Pipeline

The dataset is a collection of English-Spanish sentence pairs from [Tatoeba](https://tatoeba.org/en/downloads), a community-contributed corpus of translated sentences. Each line in the file contains an English sentence and its Spanish translation, separated by a tab character.

**Preprocessing steps:**
1. **Lowercasing** and whitespace normalization
2. **Accent removal** (a, e, etc.) to reduce vocabulary size
3. **Non-alphabetic character removal** (numbers, punctuation stripped)
4. **Special tokens**: `<sos>` (start-of-sequence) and `<eos>` (end-of-sequence) are prepended/appended

**Vocabulary building:** Words are mapped to integer indices sorted by frequency (most common = lowest index), with `<pad>=0` and `<unk>=1` reserved.

**DataLoader:** Sequences are truncated to `MAX_SEQ_LEN=128` and padded to equal length within each batch using `<pad>` tokens. Batch size: 64, shuffled for training.

In [ ]:
PATH = '../DB/spanish_english/eng-spa.txt'

In [ ]:
with open(PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()
eng_spa_pairs = [line.strip().split('\t') for line in lines if '\t' in line]

In [ ]:
eng_spa_pairs[:10]

[['Go.', 'Ve.'],
 ['Go.', 'Vete.'],
 ['Go.', 'Vaya.'],
 ['Go.', 'Váyase.'],
 ['Go.', 'Id.'],
 ['Go.', 'Vayan.'],
 ['Go.', 'Váyanse.'],
 ['Hi.', 'Hola.'],
 ['Run!', '¡Corre!'],
 ['Run!', '¡Corran!']]

In [ ]:
eng_sentences = [pair[0] for pair in eng_spa_pairs]
spa_sentences = [pair[1] for pair in eng_spa_pairs]

In [ ]:
print(eng_sentences[:10])
print(spa_sentences[:10])


['Go.', 'Go.', 'Go.', 'Go.', 'Go.', 'Go.', 'Go.', 'Hi.', 'Run!', 'Run!']
['Ve.', 'Vete.', 'Vaya.', 'Váyase.', 'Id.', 'Vayan.', 'Váyanse.', 'Hola.', '¡Corre!', '¡Corran!']


In [ ]:
def preprocess_sentence(sentence):

    sentence = sentence.lower().strip()
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[á]+", "a", sentence)
    sentence = re.sub(r"[é]+", "e", sentence)
    sentence = re.sub(r"[í]+", "i", sentence)
    sentence = re.sub(r"[ó]+", "o", sentence)
    sentence = re.sub(r"[ú]+", "u", sentence)
    sentence = re.sub(r"[^a-z]+", " ", sentence)
    sentence = sentence.strip()
    sentence = '<sos> ' + sentence + ' <eos>'
    return sentence

In [ ]:
s1 = '¿Hola @ cómo estás? 123'

In [ ]:
print(s1)
print(preprocess_sentence(s1))

¿Hola @ cómo estás? 123
<sos> hola como estas <eos>


In [ ]:
eng_sentences = [preprocess_sentence(sentence) for sentence in eng_sentences]
spa_sentences = [preprocess_sentence(sentence) for sentence in spa_sentences]

In [ ]:
spa_sentences[:10]

['<sos> ve <eos>',
 '<sos> vete <eos>',
 '<sos> vaya <eos>',
 '<sos> vayase <eos>',
 '<sos> id <eos>',
 '<sos> vayan <eos>',
 '<sos> vayanse <eos>',
 '<sos> hola <eos>',
 '<sos> corre <eos>',
 '<sos> corran <eos>']

In [ ]:
def build_vocab(sentences):

    words = [word for sentence in sentences for word in sentence.split()]
    word_count = Counter(words)
    sorted_word_counts = sorted(word_count.items(), key=lambda x:x[1], reverse=True)
    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_word_counts, 2)}
    word2idx['<pad>'] = 0
    word2idx['<unk>'] = 1
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

In [ ]:
eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
spa_word2idx, spa_idx2word = build_vocab(spa_sentences)
eng_vocab_size = len(eng_word2idx)
spa_vocab_size = len(spa_word2idx)

In [ ]:
print(eng_vocab_size, spa_vocab_size)

13877 26987


In [ ]:
class EngSpaDataset(Dataset):
    """PyTorch Dataset that converts preprocessed sentence pairs into index tensors."""
    def __init__(self, eng_sentences, spa_sentences, eng_word2idx, spa_word2idx):
        self.eng_sentences = eng_sentences
        self.spa_sentences = spa_sentences
        self.eng_word2idx = eng_word2idx
        self.spa_word2idx = spa_word2idx

    def __len__(self):
        return len(self.eng_sentences)

    def __getitem__(self, idx):
        eng_sentence = self.eng_sentences[idx]
        spa_sentence = self.spa_sentences[idx]
        # return tokens idxs
        eng_idxs = [self.eng_word2idx.get(word, self.eng_word2idx['<unk>']) for word in eng_sentence.split()]
        spa_idxs = [self.spa_word2idx.get(word, self.spa_word2idx['<unk>']) for word in spa_sentence.split()]

        return torch.tensor(eng_idxs), torch.tensor(spa_idxs)

In [ ]:
def collate_fn(batch):

    eng_batch, spa_batch = zip(*batch)
    eng_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in eng_batch]
    spa_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spa_batch]
    eng_batch = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=0)
    spa_batch = torch.nn.utils.rnn.pad_sequence(spa_batch, batch_first=True, padding_value=0)
    return eng_batch, spa_batch


### Training

The training loop uses **teacher forcing**: at each decoder time step, the ground-truth previous token is provided as input (rather than the model's own prediction). This enables efficient parallel computation of the loss across all positions.

**Key details:**
- **Loss function**: `CrossEntropyLoss(ignore_index=0)` — standard classification loss over the target vocabulary, ignoring `<pad>` tokens (index 0) so padding doesn't affect gradients
- **Target shifting**: The decoder input is `<sos> w1 w2 ... wN` and the expected output is `w1 w2 ... wN <eos>` (shifted by one position)
- **Optimizer**: Adam with learning rate 0.0001
- **Epochs**: 10 passes over the full dataset

In [ ]:
def train(model, dataloader, loss_function, optimiser, epochs):

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for i, (eng_batch, spa_batch) in enumerate(dataloader):
            eng_batch = eng_batch.to(device)
            spa_batch = spa_batch.to(device)
            # Teacher forcing: input is all tokens except last, output is all except first
            # e.g. input: <sos> w1 w2 w3 -> predict: w1 w2 w3 <eos>
            target_input = spa_batch[:, :-1]
            target_output = spa_batch[:, 1:].contiguous().view(-1)
            # Zero grads
            optimiser.zero_grad()
            # run model
            output = model(eng_batch, target_input)
            output = output.view(-1, output.size(-1))
            # loss\
            loss = loss_function(output, target_output)
            # gradient and update parameters
            loss.backward()
            optimiser.step()
            total_loss += loss.item()

        avg_loss = total_loss/len(dataloader)
        print(f'Epoch: {epoch}/{epochs}, Loss: {avg_loss:.4f}')



In [ ]:
BATCH_SIZE = 64
dataset = EngSpaDataset(eng_sentences, spa_sentences, eng_word2idx, spa_word2idx)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [ ]:
model = Transformer(d_model=512, num_heads=8, d_ff=2048, num_layers=6,
                    input_vocab_size=eng_vocab_size, target_vocab_size=spa_vocab_size,
                    max_len=MAX_SEQ_LEN, dropout=0.1)

In [ ]:
model = model.to(device)
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimiser = optim.Adam(model.parameters(), lr=0.0001)


In [ ]:
train(model, dataloader, loss_function, optimiser, epochs = 10)

Epoch: 0/10, Loss: 3.6556
Epoch: 1/10, Loss: 2.1598
Epoch: 2/10, Loss: 1.6059
Epoch: 3/10, Loss: 1.2497
Epoch: 4/10, Loss: 0.9863
Epoch: 5/10, Loss: 0.7816
Epoch: 6/10, Loss: 0.6208
Epoch: 7/10, Loss: 0.5001
Epoch: 8/10, Loss: 0.4157
Epoch: 9/10, Loss: 0.3601


### Inference: Autoregressive Translation

During inference (translation), the model generates the target sentence **one token at a time** using **greedy autoregressive decoding**:

1. The English source sentence is encoded once by the encoder
2. The decoder starts with just the `<sos>` token
3. At each step, the decoder predicts the next token by taking the **argmax** of the output logits (greedy search)
4. The predicted token is appended to the decoder input, and the process repeats
5. Generation stops when the model produces `<eos>` or reaches `MAX_SEQ_LEN`

This differs from training, where teacher forcing provides the ground-truth tokens. During inference, the model must rely on its own predictions, so errors can compound (exposure bias).

In [ ]:
def sentence_to_indices(sentence, word2idx):

    return [word2idx.get(word, word2idx['<unk>']) for word in sentence.split()]

def indices_to_sentence(indices, idx2word):

    return ' '.join([idx2word[idx] for idx in indices if idx in idx2word and idx2word[idx] != '<pad>'])

def translate_sentence(model, sentence, eng_word2idx, spa_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):

    model.eval()
    sentence = preprocess_sentence(sentence)
    input_indices = sentence_to_indices(sentence, eng_word2idx)
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(device)

    # Start decoding with the <sos> (start-of-sequence) token
    tgt_indices = [spa_word2idx['<sos>']]
    tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_tensor, tgt_tensor)
            output = output.squeeze(0)
            # Greedy decoding: pick the token with highest probability
            next_token = output.argmax(dim=-1)[-1].item()
            tgt_indices.append(next_token)
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            if next_token == spa_word2idx['<eos>']:
                break

    return indices_to_sentence(tgt_indices, spa_idx2word)

In [ ]:
def evaluate_translations(model, sentences, eng_word2idx, spa_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):

    for sentence in sentences:
        translation = translate_sentence(model, sentence, eng_word2idx, spa_word2idx, spa_idx2word, max_len, device)
        print(f'Input: {sentence}')
        print(f'Translation: {translation}')
        print()

# 15 test sentences covering varied topics and sentence structures
test_sentences = [
    "Hello, how are you?",
    "I am learning artificial intelligence.",
    "Artificial intelligence is great.",
    "Good night!",
    "Good morning!",
    "What is your name?",
    "I love to read books.",
    "The weather is nice today.",
    "How much does this cost?",
    "Thank you very much.",
    "Where is the bathroom?",
    "I need help.",
    "Have a nice day!",
    "I am a student.",
    "She is my friend.",
]

# Use the same device as training (supports CUDA, MPS, and CPU)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

model = model.to(device)

evaluate_translations(model, test_sentences, eng_word2idx, spa_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device=device)


Input: Hello, how are you?
Translation: <sos> como estas sois <eos>

Input: I am learning artificial intelligence.
Translation: <sos> estoy aprendiendo inteligencia artificial <eos>

Input: Artificial intelligence is great.
Translation: <sos> el artificial de inteligencia es genial <eos>

Input: Good night!
Translation: <sos> buenas noches <eos>

Input: Good morning!
Translation: <sos> buenos dias <eos>

Input: What is your name?
Translation: <sos> como te llamas <eos>

Input: I love to read books.
Translation: <sos> me encanta leer libros <eos>

Input: The weather is nice today.
Translation: <sos> hoy hace buen tiempo <eos>

Input: How much does this cost?
Translation: <sos> cuanto cuesta esto <eos>

Input: Thank you very much.
Translation: <sos> muchisimas gracias <eos>

Input: Where is the bathroom?
Translation: <sos> donde esta el ba o <eos>

Input: I need help.
Translation: <sos> necesito ayuda <eos>

Input: Have a nice day!
Translation: <sos> ten un buen dia <eos>

Input: I am a 

## Conclusions

### What We Learned

Through this activity, we gained hands-on understanding of the Transformer architecture by implementing an English-to-Spanish translator from scratch in PyTorch. Key takeaways include:

- **Self-attention replaces recurrence**: Unlike RNNs/LSTMs, the Transformer processes all positions simultaneously through attention, enabling parallelization during training and better capturing long-range dependencies.
- **Multi-head attention** allows the model to jointly attend to information from different representation subspaces at different positions. With 8 heads and d_model=512, each head operates on a 64-dimensional subspace.
- **Positional encoding** is essential since the self-attention mechanism is inherently position-agnostic. The sinusoidal functions at different frequencies allow the model to learn relative positions.
- **Masking** serves dual purposes: padding masks ignore `<pad>` tokens in variable-length batches, while the causal mask ensures the decoder cannot "cheat" by looking at future tokens during autoregressive generation.
- **Teacher forcing** during training (feeding ground-truth previous tokens to the decoder) enables efficient parallel training, even though inference must be done sequentially token by token.

### Training Results

The model trained for 10 epochs on 144,215 sentence pairs, with the loss decreasing steadily from **3.66 to 0.36**:

| Epoch | Loss |
|-------|------|
| 0 | 3.6556 |
| 1 | 2.1598 |
| 2 | 1.6059 |
| 3 | 1.2497 |
| 4 | 0.9863 |
| 5 | 0.7816 |
| 6 | 0.6208 |
| 7 | 0.5001 |
| 8 | 0.4157 |
| 9 | 0.3601 |

The loss was still decreasing at epoch 9, suggesting that additional epochs could further improve translation quality.

### Observations on Translation Quality

Out of 15 test sentences, the model produced surprisingly strong results. Roughly **12 out of 15 translations were accurate or near-perfect**:

- **Perfect or near-perfect translations**: "Good night!" → *buenas noches*, "Good morning!" → *buenos dias*, "What is your name?" → *como te llamas*, "I love to read books." → *me encanta leer libros*, "The weather is nice today." → *hoy hace buen tiempo*, "How much does this cost?" → *cuanto cuesta esto*, "I need help." → *necesito ayuda*, "I am a student." → *soy estudiante*, "She is my friend." → *ella es mi amiga*, "Thank you very much." → *muchisimas gracias*
- **Good with minor issues**: "Hello, how are you?" → *como estas sois* (correct start, spurious extra word "sois"), "Have a nice day!" → *ten un buen dia* (reasonable)
- **Partially correct**: "Artificial intelligence is great." → *el artificial de inteligencia es genial* (word order shuffled, extra articles)
- **Character handling issue**: "Where is the bathroom?" → *donde esta el ba o* — the word "baño" was split because the ñ was stripped during preprocessing, showing a limitation of the accent-removal approach

The model clearly learned meaningful English-to-Spanish mappings. It performs best on short to medium-length sentences with common vocabulary and straightforward structure.

### Limitations

1. **Word-level tokenization**: Each unique word form is a separate token, leading to a large vocabulary (~14K English, ~27K Spanish) with many rare words. Subword tokenization (BPE, WordPiece) would reduce vocabulary size and handle unseen words better.
2. **Greedy decoding**: Always selecting the highest-probability token can miss better overall translations. Beam search would explore multiple candidate sequences.
3. **Loss still decreasing**: At 0.36 after 10 epochs, the model had not fully converged. More training time would likely improve results.
4. **No learning rate scheduling**: The original paper uses a warmup schedule; a constant learning rate of 0.0001 may lead to suboptimal convergence.
5. **Accent and special character removal**: Preprocessing strips accents and characters like ñ, which causes issues (e.g., "baño" → "ba o") and loses semantic distinctions (e.g., "si" meaning "if" vs "si" meaning "yes" from "sí").

### Possible Improvements

- **Subword tokenization** (e.g., Byte Pair Encoding) to handle rare and compound words
- **Beam search decoding** for higher-quality translations
- **Learning rate warmup** following the original paper's schedule: `lr = d_model^(-0.5) * min(step^(-0.5), step * warmup_steps^(-1.5))`
- **More training epochs** or a larger dataset
- **BLEU score evaluation** for quantitative measurement of translation quality
- **Attention visualization** to understand what the model focuses on during translation